In [4]:
def multiplicative_order(k, n):
    if n == 1:
        return 0
    m = k
    for i in range(1, n):
        if m % n == 1:
            return i
        m = m*k
# multiplicative_order(2, 7)
assert multiplicative_order(2, 7) == 3
assert multiplicative_order(2,5) == 4
assert multiplicative_order(2,1) == 0

In [5]:
def split_the_2_part(n):
    two_part = 2^(n.valuation(2))
    return two_part, n// two_part

In [6]:
def hannukah_criterion(n):
    two_part, p = split_the_2_part(n)
    if not p.is_prime() or two_part != 2:
        return false
    o = multiplicative_order(2, p)
    return all(
        d % (2*o) != two_part - 1
        for d in
        divisors(n - 1)[1:-1]
    )

def wtf(n):
    if not n.is_prime():
        return false
        
assert not hannukah_criterion(28)
assert hannukah_criterion(10)
assert not hannukah_criterion(8)
assert not hannukah_criterion(16)
assert not hannukah_criterion(44)
assert hannukah_criterion(6)

In [17]:
def has_solution(target, coefficients):
    MILP = MixedIntegerLinearProgram()
    w = MILP.new_variable(integer=True, nonnegative=True)
    # print(target)
    # print(coefficients)
    MILP.add_constraint(
        sum( c*w[i] for i,c in enumerate(coefficients)) == target
    )
    # MILP.show()
    return any(true for _ in MILP.polyhedron().integral_points())

assert has_solution(10, (5,))
assert has_solution(18, (5,2))
assert not has_solution(6, (5,4))

In [22]:
def block_fixed_at_2_criterion(n):
    # print(n)
    two_part, odd_part = split_the_2_part(n)
    if odd_part == 1 or two_part == 2:
        return false
    orders = tuple(multiplicative_order(2, d) for d in divisors(odd_part)[1:])
    return not any(
        has_solution(
            d - two_part + 1,
            tuple(two_part*o for o in orders)
        )
        for d in divisors(n - 1)[1:-1]
    )
    
assert block_fixed_at_2_criterion(10)
assert block_fixed_at_2_criterion(22)
assert block_fixed_at_2_criterion(50)
assert not block_fixed_at_2_criterion(9026)
assert block_fixed_at_2_criterion(40)
assert not block_fixed_at_2_criterion(52)
assert not block_fixed_at_2_criterion(16)

AssertionError: 

In [21]:
def unsolved_cases_up_to(N):
    return (
        n for n in map(ZZ, range(3, N + 1)) if not any((
            n % 2 == 1, 
            (n - 1).is_prime(),
            # hannukah_criterion(n),
            block_fixed_at_2_criterion(n),
            # n in (16,28,36,52,64) #factorization mod (n-2)/2
        ))
    )

list(unsolved_cases_up_to(100))
# len(list(unsolved_cases_up_to(1000)))

[16, 28, 36, 40, 52, 56, 64, 66, 70, 76, 88, 92, 96, 100]

In [32]:
S.<y> = PolynomialRing(QQ)
# all((y^(11*i) + 11*y + 1).discriminant().valuation(3) <= 3*i + 1 for i in range(1,100) )
for a in range(2,50):
    for p in ZZ(a).prime_divisors():
        for i in range(2, 50):
            if a % 2 == 1 and ZZ(a).is_squarefree() and (y^(p*i) + a*y + 1).discriminant().valuation(p) > p*i + 1:
                print((a,p,p*i,(y^(p*i) + a*y + 1).discriminant().valuation(p)))
# 20.prime_divisors()

In [133]:
S.<y> = PolynomialRing(QQ)
def validate_case(n,a,p):
    if ZZ(a).valuation(p) != ZZ(n).valuation(p):
        return True
    v = ZZ(a).valuation(p)
    p_part = p ** v
    n0 = n / p_part
    a0 = a / p_part
    if (GF(p)(n0) / GF(p)(a0))**n0 + 1 % p == 0:
        # if ZZ(a).is_squarefree() and (y^n + a*y + 1).discriminant().valuation(p) != n * v + 1:
        #     print((n,a,p,(y^n + a*y + 1).discriminant().valuation(p)))
        return True
    return (y^n + a*y + 1).discriminant().valuation(p) == n * v

bound = 100

tuple(
    (n,a,p,(y^n + a*y + 1).discriminant().valuation(p))
    for n in range(bound)
    for a in range(-bound,bound) if a != 0
    for p in ZZ(a).prime_divisors()
    if not validate_case(n,a,p)
)

()

In [134]:
S.<y> = PolynomialRing(QQ)
bound = 23

tuple(
    (n,a,p,(y^n + a*y + 1).discriminant().valuation(p))
    for n in range(25)
    for a in range(-bound,bound) if abs(a) > 2
    for p in ZZ(a).prime_divisors()
    if ZZ(a).prime_divisors() == ZZ((y^n + a*y + 1).discriminant()).prime_divisors()
)

((3, -3, 3, 4),)

In [51]:
solve(x^3-3*x+1==0,x)

[x == -1/2*(I*sqrt(3) + 1)*(1/2*I*sqrt(3) - 1/2)^(1/3) + (1/2*I*sqrt(3) - 1/2)^(2/3),
 x == (1/2*I*sqrt(3) - 1/2)^(4/3) - 1/2*(I*sqrt(3) + 1)/(1/2*I*sqrt(3) - 1/2)^(1/3),
 x == (1/2*I*sqrt(3) - 1/2)^(1/3) + 1/(1/2*I*sqrt(3) - 1/2)^(1/3)]

In [34]:
factor(2^18 + 6)

2 * 5^2 * 7^2 * 107

In [33]:
# 1. Define polynomial and number field
x = polygen(QQ, 'x')
f = x^4 + 2*x + 2
K.<a> = NumberField(f)

# 2. Get Galois closure and Galois group
L.<b> = K.galois_closure()
G = L.galois_group()

# 3. Choose a prime and find primes above it
p = 2  # The polynomial x^3-2 is ramified at 3
primes = L.primes_above(p)
Q = primes[0]  # Take the first prime ideal

# 4. Compute decomposition group (Inertia group D0)
D0 = G.decomposition_group(Q)
print(f"Inertia group (Order): {D0.order()}")

# 5. Compute higher ramification groups (V0, V1, ... Vi)
# G.ramification_group(Q, i)
V0 = G.ramification_group(Q, 0)
V1 = G.ramification_group(Q, 1)

print(f"V0 (Inertia): {V0.order()}")
print(f"V1 (Wild Ramification): {V1.order()}")


Inertia group (Order): 24
V0 (Inertia): 12
V1 (Wild Ramification): 4


In [2]:
S.<y> = PolynomialRing(QQ)
# (x^2 - 17).factor()
K.<alpha> = NumberField(y^10 + 2*y + 2, 'a')
K.galois_group()

Galois group 10T45 (S10) with order 3628800 of y^10 + 2*y + 2

In [8]:
S.<y> = PolynomialRing(GF(84223))
factor((y^8 - 2*y + 1)/(y-1))

(y + 23781) * (y + 36095)^2 * (y^4 + 72476*y^3 + 10064*y^2 + 84052*y + 52791)